# 64 — W4 dev gate: config 170 (ensemble + SID) vs config 132 (current champion)

Runs both configs on the dev split, computes per-turn nDCG@20, then runs
paired-bootstrap CI (n_resamples=1000, alpha=0.05). Gate passes if:
  - mean Δ nDCG@20 ≥ +0.005, AND
  - paired-bootstrap CI lower bound > 0 (statistically real improvement).

Wallclock: ~2.5-3 hr on L4 (two diagnostic runs back-to-back).


In [ ]:
# 1) Setup: clone + auth + Drive + deps. (Same pattern as notebook 62.)
import os
from google.colab import userdata, drive
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
drive.mount('/content/drive', force_remount=False)

BRANCH = 'fresh-model'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

DRIVE_BASE = '/content/drive/MyDrive'
LOCAL_BASE = '/content/recsys2026/experiments/cache'
os.makedirs(LOCAL_BASE, exist_ok=True)
for name, drive_subdir in [
    ('sid', 'recsys2026_sid_cache'),
    ('sid_training', 'recsys2026_sid_training_cache'),
    ('dense', 'recsys2026_dense_cache'),
]:
    src = f'{DRIVE_BASE}/{drive_subdir}'
    dst = f'{LOCAL_BASE}/{name}'
    os.makedirs(src, exist_ok=True)
    if os.path.islink(dst): os.unlink(dst)
    elif os.path.exists(dst):
        import shutil; shutil.rmtree(dst)
    os.symlink(src, dst)

!pip install -q -U "peft>=0.10" "transformers>=4.40" "accelerate>=0.30" "torchao>=0.17"


In [ ]:
# 2) Run config 170 (ensemble + SID) on the dev split.
# Note: --tid is the config name (without .yaml); --eval_dataset='dev' triggers
# the run on the dev split (the dataset itself is set by test_dataset_name in
# the config YAML). Output auto-saved to music-crs-baselines/exp/inference/dev/<tid>.json.
%cd /content/recsys2026/music-crs-baselines
!python run_inference_blindset.py \
    --tid 170-wrrf-sid-v5kto-blindsetA \
    --eval_dataset dev \
    --batch_size 32 \
    2>&1 | tail -50

In [ ]:
# 3) Run config 132 (current champion, no SID) on the same dev split for paired-bootstrap.
!python run_inference_blindset.py \
    --tid 132-bge-m3-v5kto-prorank-rerank-blindsetA \
    --eval_dataset dev \
    --batch_size 32 \
    2>&1 | tail -50

In [ ]:
# 4) Paired-bootstrap CI: 170 vs 132 on dev. Uses scripts/compare_blind_predictions.py
# which reads the two prediction.json files + dev gold + emits the gate JSON.
%cd /content/recsys2026
!python scripts/compare_blind_predictions.py \
    --pred_a music-crs-baselines/exp/inference/dev/170-wrrf-sid-v5kto-blindsetA.json \
    --pred_b music-crs-baselines/exp/inference/dev/132-bge-m3-v5kto-prorank-rerank-blindsetA.json \
    --dataset talkpl-ai/TalkPlayData-Challenge-Dataset \
    --gold_split test \
    --label_a 'wRRF+SID (170)' \
    --label_b 'wRRF current (132)' \
    --n_resamples 1000 \
    --alpha 0.05 \
    --output experiments/diagnostic_runs/w4_gate.json

In [ ]:
# 5) Display gate decision.
import json
m = json.load(open('experiments/diagnostic_runs/w4_gate.json'))
print(json.dumps({k: v for k, v in m.items() if k != 'per_session'}, indent=2))
print()
print('=' * 60)
delta = m.get('delta_mean', 0)
ci_lo = m.get('paired_bootstrap_ci', {}).get('lo', -1)
gate_pass = delta >= 0.005 and ci_lo > 0
if gate_pass:
    print(f"GATE PASS — Δ nDCG@20 = {delta:+.4f}, CI lo = {ci_lo:+.4f} > 0")
    print('PROCEED to Blind-A submission via notebook 63.')
else:
    print(f"GATE FAIL — Δ nDCG@20 = {delta:+.4f}, CI lo = {ci_lo:+.4f}")
    print('Required: Δ ≥ +0.005 AND CI lo > 0. Do NOT submit Blind-A yet.')
print('=' * 60)

## After the gate

**Gate PASS**: open notebook 63 and run config 170 on Blind-A for the official submission.

**Gate FAIL**: do not submit Blind-A. Options:
  - **Δ small but CI excludes 0 (e.g. Δ=+0.003)**: tighten in W5 by tuning SID weight (sweep {0.3, 0.5, 0.7, 1.0}); the SID stream may need a smaller weight to not dominate.
  - **Δ near 0 or negative**: SID is hurting the ensemble. Try config 171 (pure-SID) to isolate whether SID alone is better or worse than current wRRF — informs whether to keep SID at all.
  - **High CI variance**: dev sample may be too small. Re-run on full dev split or wait until W5 for Blind-A signal.
